# Notebook 06 — CRF Baseline (local, CPU)

**Objetivo:** Treinar e avaliar o modelo CRF como baseline do experimento.
Roda inteiramente local (CPU), sem GPU.

**Dados:** `data/annotated/train.conll`, `dev.conll`, `test.conll`  
**Saída:** `outputs/models/crf/model.pkl` e `outputs/results/crf/metrics.json`

**Referência:** Lafferty et al. (2001) — CRF; sklearn-crfsuite

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ANNOTATED_DIR = PROJECT_ROOT / "data" / "annotated"
MODELS_DIR    = PROJECT_ROOT / "outputs" / "models" / "crf"
RESULTS_DIR   = PROJECT_ROOT / "outputs" / "results" / "crf"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dados anotados: {ANNOTATED_DIR}")
for f in ["train.conll", "dev.conll", "test.conll"]:
    status = "✓" if (ANNOTATED_DIR / f).exists() else "✗ NÃO ENCONTRADO"
    print(f"  {status} {f}")

In [ ]:
# ── Carregar dados ────────────────────────────────────────────────────────────
from src.ner.trainer import load_conll

train_data = load_conll(str(ANNOTATED_DIR / "train.conll"))
dev_data   = load_conll(str(ANNOTATED_DIR / "dev.conll"))
test_data  = load_conll(str(ANNOTATED_DIR / "test.conll"))

print(f"Treino:    {len(train_data):>5,} sentenças")
print(f"Validação: {len(dev_data):>5,} sentenças")
print(f"Teste:     {len(test_data):>5,} sentenças")

In [ ]:
# ── Treinar CRF ───────────────────────────────────────────────────────────────
from src.ner.trainer import CRFTrainer

trainer = CRFTrainer()
trainer.train(train_data, dev_data=dev_data)

# Salvar modelo
trainer.save(MODELS_DIR / "model.pkl")

In [ ]:
# ── Avaliar no conjunto de teste ──────────────────────────────────────────────
from src.evaluation.metrics import NERMetrics
import json
import pandas as pd

X_test = [tokens for tokens, _ in test_data]
y_true = [labels for _, labels in test_data]
y_pred = trainer.predict(X_test)

metrics = NERMetrics.compute_seqeval(y_true, y_pred)
print(f"\n=== Resultados CRF no conjunto de Teste ===")
print(f"Precision : {metrics['overall_precision']:.4f}")
print(f"Recall    : {metrics['overall_recall']:.4f}")
print(f"F1        : {metrics['overall_f1']:.4f}")

# Salvar métricas
with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"\n✓ Métricas salvas em {RESULTS_DIR / 'metrics.json'}")

In [ ]:
# ── Detalhamento por entidade ─────────────────────────────────────────────────
if "per_entity" in metrics and metrics["per_entity"]:
    df_per_entity = pd.DataFrame(metrics["per_entity"]).T
    df_per_entity = df_per_entity[["precision", "recall", "f1-score", "support"]]
    df_per_entity = df_per_entity.sort_values("f1-score", ascending=False)
    display(df_per_entity.style.format({
        "precision": "{:.4f}", "recall": "{:.4f}", "f1-score": "{:.4f}"
    }))